In [17]:
import os

PROJECT = "/content/warp_reduction"
SRC = f"{PROJECT}/src"
RESULTS = f"{PROJECT}/results"

os.makedirs(SRC, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

In [18]:
%%writefile /content/warp_reduction/src/scalable_reduction.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <algorithm>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << cudaGetErrorString(err) << "\n"; \
        return 1; \
    } \
} while (0)

constexpr int THREADS = 256;
constexpr int MAX_BLOCKS = 1024;

// Shared-memory block reduction.
__global__ void reduceShared(
    const int* input, int* partial, int n
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;
    for (; i < n; i += step) {
        sum += input[i];
    }

    s[tid] = sum;
    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0; stride /= 2) {
        if (tid < stride) {
            s[tid] += s[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        partial[blockIdx.x] = s[0];
    }
}

// Warp-shuffle block reduction.
__global__ void reduceShuffle(
    const int* input, int* partial, int n
) {
    __shared__ int warpSums[THREADS / 32];

    int tid = threadIdx.x;
    int lane = tid % 32;
    int warp = tid / 32;
    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;
    for (; i < n; i += step) {
        sum += input[i];
    }

    unsigned int mask = 0xffffffff;
    for (int offset = 16; offset > 0; offset /= 2) {
        sum += __shfl_down_sync(mask, sum, offset);
    }

    if (lane == 0) {
        warpSums[warp] = sum;
    }
    __syncthreads();

    // First warp reduces the 8 warp totals.
    if (warp == 0) {
        int total = (lane < THREADS / 32)
            ? warpSums[lane] : 0;

        for (int offset = 16; offset > 0; offset /= 2) {
            total += __shfl_down_sync(
                mask, total, offset
            );
        }

        if (lane == 0) {
            partial[blockIdx.x] = total;
        }
    }
}

// Final stage: one block reduces block partial sums.
__global__ void finishShared(
    const int* partial, int* output, int count
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int sum = 0;

    for (int i = tid; i < count; i += blockDim.x) {
        sum += partial[i];
    }

    s[tid] = sum;
    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0; stride /= 2) {
        if (tid < stride) {
            s[tid] += s[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) output[0] = s[0];
}

int main() {
    std::vector<int> sizes = {
        32, 256, 1000, 65536, 1000000
    };

    for (int n : sizes) {

        std::vector<int> h_input(n);
        for (int i = 0; i < n; i++) {
            h_input[i] = (i % 10) + 1;
        }

        long long expected = std::accumulate(
            h_input.begin(), h_input.end(), 0LL
        );

        int blocks = std::min(
            (n + THREADS - 1) / THREADS,
            MAX_BLOCKS
        );

        int *d_input = nullptr;
        int *d_partial = nullptr;
        int *d_output = nullptr;

        CHECK(cudaMalloc(
            &d_input, n * sizeof(int)
        ));
        CHECK(cudaMalloc(
            &d_partial, blocks * sizeof(int)
        ));
        CHECK(cudaMalloc(
            &d_output, sizeof(int)
        ));
        CHECK(cudaMemcpy(
            d_input, h_input.data(),
            n * sizeof(int), cudaMemcpyHostToDevice
        ));

        int sharedResult = 0;
        int shuffleResult = 0;

        // Shared-memory first stage + shared final stage.
        reduceShared<<<blocks, THREADS>>>(
            d_input, d_partial, n
        );
        CHECK(cudaGetLastError());
        finishShared<<<1, THREADS>>>(
            d_partial, d_output, blocks
        );
        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());
        CHECK(cudaMemcpy(
            &sharedResult, d_output, sizeof(int),
            cudaMemcpyDeviceToHost
        ));

        // Shuffle first stage + same final stage.
        reduceShuffle<<<blocks, THREADS>>>(
            d_input, d_partial, n
        );
        CHECK(cudaGetLastError());
        finishShared<<<1, THREADS>>>(
            d_partial, d_output, blocks
        );
        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());
        CHECK(cudaMemcpy(
            &shuffleResult, d_output, sizeof(int),
            cudaMemcpyDeviceToHost
        ));

        std::cout
            << "N=" << n
            << " expected=" << expected
            << " shared=" << sharedResult
            << " shuffle=" << shuffleResult
            << " "
            << ((sharedResult == expected &&
                 shuffleResult == expected)
                ? "PASS" : "FAIL")
            << "\n";

        CHECK(cudaFree(d_input));
        CHECK(cudaFree(d_partial));
        CHECK(cudaFree(d_output));
    }

    return 0;
}

Writing /content/warp_reduction/src/scalable_reduction.cu


In [19]:
!nvcc /content/warp_reduction/src/scalable_reduction.cu \
    -o /content/warp_reduction/src/scalable_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!/content/warp_reduction/src/scalable_reduction

N=32 expected=168 shared=168 shuffle=168 PASS
N=256 expected=1396 shared=1396 shuffle=1396 PASS
N=1000 expected=5500 shared=5500 shuffle=5500 PASS
N=65536 expected=360436 shared=360436 shuffle=360436 PASS
N=1000000 expected=5500000 shared=5500000 shuffle=5500000 PASS


In [21]:

%%writefile /content/warp_reduction/src/scalable_reduction.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <algorithm>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) << "\n"; \
        return 1; \
    } \
} while (0)

constexpr int THREADS = 256;
constexpr int MAX_BLOCKS = 1024;

// Shared-memory block reduction.
__global__ void reduceShared(
    const int* input, int* partial, int n
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;
    for (; i < n; i += step) {
        sum += input[i];
    }

    s[tid] = sum;
    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0; stride /= 2) {
        if (tid < stride) {
            s[tid] += s[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        partial[blockIdx.x] = s[0];
    }
}

// Warp-shuffle block reduction.
__global__ void reduceShuffle(
    const int* input, int* partial, int n
) {
    __shared__ int warpSums[THREADS / 32];

    int tid = threadIdx.x;
    int lane = tid % 32;
    int warp = tid / 32;

    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;
    for (; i < n; i += step) {
        sum += input[i];
    }

    unsigned int mask = 0xffffffff;

    // Reduce values inside each warp.
    for (int offset = 16; offset > 0; offset /= 2) {
        sum += __shfl_down_sync(mask, sum, offset);
    }

    // Lane 0 of each warp writes its warp sum.
    if (lane == 0) {
        warpSums[warp] = sum;
    }
    __syncthreads();

    // First warp reduces the 8 warp totals.
    if (warp == 0) {
        int total = (lane < THREADS / 32)
            ? warpSums[lane] : 0;

        for (int offset = 16; offset > 0; offset /= 2) {
            total += __shfl_down_sync(
                mask, total, offset
            );
        }

        if (lane == 0) {
            partial[blockIdx.x] = total;
        }
    }
}

// Final stage: one block reduces block partial sums.
__global__ void finishShared(
    const int* partial, int* output, int count
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int sum = 0;

    for (int i = tid; i < count; i += blockDim.x) {
        sum += partial[i];
    }

    s[tid] = sum;
    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0; stride /= 2) {
        if (tid < stride) {
            s[tid] += s[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        output[0] = s[0];
    }
}

int main() {
    std::vector<int> sizes = {
        32, 256, 1000, 65536, 1000000
    };

    const int ITERS = 500;

    for (int n : sizes) {
        std::vector<int> h_input(n);

        for (int i = 0; i < n; i++) {
            h_input[i] = (i % 10) + 1;
        }

        long long expected = std::accumulate(
            h_input.begin(), h_input.end(), 0LL
        );

        int blocks = std::min(
            (n + THREADS - 1) / THREADS,
            MAX_BLOCKS
        );

        int *d_input = nullptr;
        int *d_partial = nullptr;
        int *d_output = nullptr;

        CHECK(cudaMalloc(
            &d_input, n * sizeof(int)
        ));
        CHECK(cudaMalloc(
            &d_partial, blocks * sizeof(int)
        ));
        CHECK(cudaMalloc(
            &d_output, sizeof(int)
        ));

        CHECK(cudaMemcpy(
            d_input, h_input.data(),
            n * sizeof(int),
            cudaMemcpyHostToDevice
        ));

        int sharedResult = 0;
        int shuffleResult = 0;

        // ------------------------------
        // 1. Correctness: shared method
        // ------------------------------
        reduceShared<<<blocks, THREADS>>>(
            d_input, d_partial, n
        );
        CHECK(cudaGetLastError());

        finishShared<<<1, THREADS>>>(
            d_partial, d_output, blocks
        );
        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(
            &sharedResult, d_output, sizeof(int),
            cudaMemcpyDeviceToHost
        ));

        // ------------------------------
        // 2. Correctness: shuffle method
        // ------------------------------
        reduceShuffle<<<blocks, THREADS>>>(
            d_input, d_partial, n
        );
        CHECK(cudaGetLastError());

        finishShared<<<1, THREADS>>>(
            d_partial, d_output, blocks
        );
        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(
            &shuffleResult, d_output, sizeof(int),
            cudaMemcpyDeviceToHost
        ));

        bool passed =
            (sharedResult == expected) &&
            (shuffleResult == expected);

        std::cout
            << "N=" << n
            << " expected=" << expected
            << " shared=" << sharedResult
            << " shuffle=" << shuffleResult
            << " " << (passed ? "PASS" : "FAIL")
            << "\n";

        // ------------------------------
        // 3. Warm-up both methods
        // ------------------------------
        for (int i = 0; i < 50; i++) {
            reduceShared<<<blocks, THREADS>>>(
                d_input, d_partial, n
            );
            finishShared<<<1, THREADS>>>(
                d_partial, d_output, blocks
            );

            reduceShuffle<<<blocks, THREADS>>>(
                d_input, d_partial, n
            );
            finishShared<<<1, THREADS>>>(
                d_partial, d_output, blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());

        // ------------------------------
        // 4. Create timing events
        // ------------------------------
        cudaEvent_t start, stop;
        CHECK(cudaEventCreate(&start));
        CHECK(cudaEventCreate(&stop));

        // ------------------------------
        // 5. Benchmark shared reduction
        // ------------------------------
        CHECK(cudaEventRecord(start));

        for (int i = 0; i < ITERS; i++) {
            reduceShared<<<blocks, THREADS>>>(
                d_input, d_partial, n
            );
            finishShared<<<1, THREADS>>>(
                d_partial, d_output, blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaEventRecord(stop));
        CHECK(cudaEventSynchronize(stop));

        float sharedMs = 0.0f;
        CHECK(cudaEventElapsedTime(
            &sharedMs, start, stop
        ));

        // ------------------------------
        // 6. Benchmark warp-shuffle reduction
        // ------------------------------
        CHECK(cudaEventRecord(start));

        for (int i = 0; i < ITERS; i++) {
            reduceShuffle<<<blocks, THREADS>>>(
                d_input, d_partial, n
            );
            finishShared<<<1, THREADS>>>(
                d_partial, d_output, blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaEventRecord(stop));
        CHECK(cudaEventSynchronize(stop));

        float shuffleMs = 0.0f;
        CHECK(cudaEventElapsedTime(
            &shuffleMs, start, stop
        ));

        float sharedUs =
            sharedMs * 1000.0f / ITERS;
        float shuffleUs =
            shuffleMs * 1000.0f / ITERS;

        std::cout
            << "Shared avg (us): "
            << sharedUs << "\n";

        std::cout
            << "Shuffle avg (us): "
            << shuffleUs << "\n";

        if (shuffleUs > 0.0f) {
            std::cout
                << "Shuffle speedup: "
                << sharedUs / shuffleUs
                << "x\n";
        }

        std::cout << "--------------------------\n";

        // ------------------------------
        // 7. Cleanup
        // ------------------------------
        CHECK(cudaEventDestroy(start));
        CHECK(cudaEventDestroy(stop));

        CHECK(cudaFree(d_input));
        CHECK(cudaFree(d_partial));
        CHECK(cudaFree(d_output));
    }

    return 0;
}

Overwriting /content/warp_reduction/src/scalable_reduction.cu


In [22]:
!nvcc /content/warp_reduction/src/scalable_reduction.cu -o /content/warp_reduction/src/scalable_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [23]:
!/content/warp_reduction/src/scalable_reduction

N=32 expected=168 shared=168 shuffle=168 PASS
Shared avg (us): 9.13133
Shuffle avg (us): 8.09824
Shuffle speedup: 1.12757x
--------------------------
N=256 expected=1396 shared=1396 shuffle=1396 PASS
Shared avg (us): 8.03629
Shuffle avg (us): 7.54893
Shuffle speedup: 1.06456x
--------------------------
N=1000 expected=5500 shared=5500 shuffle=5500 PASS
Shared avg (us): 8.08966
Shuffle avg (us): 7.87245
Shuffle speedup: 1.02759x
--------------------------
N=65536 expected=360436 shared=360436 shuffle=360436 PASS
Shared avg (us): 11.7883
Shuffle avg (us): 9.32614
Shuffle speedup: 1.264x
--------------------------
N=1000000 expected=5500000 shared=5500000 shuffle=5500000 PASS
Shared avg (us): 28.0576
Shuffle avg (us): 19.995
Shuffle speedup: 1.40323x
--------------------------


In [24]:
!nvcc /content/warp_reduction/src/scalable_reduction.cu \
-o /content/warp_reduction/src/scalable_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!/content/warp_reduction/src/scalable_reduction

N=32 expected=168 shared=168 shuffle=168 PASS
Shared avg (us): 6.82528
Shuffle avg (us): 6.16026
Shuffle speedup: 1.10795x
--------------------------
N=256 expected=1396 shared=1396 shuffle=1396 PASS
Shared avg (us): 6.8489
Shuffle avg (us): 6.23021
Shuffle speedup: 1.0993x
--------------------------
N=1000 expected=5500 shared=5500 shuffle=5500 PASS
Shared avg (us): 6.91405
Shuffle avg (us): 6.34694
Shuffle speedup: 1.08935x
--------------------------
N=65536 expected=360436 shared=360436 shuffle=360436 PASS
Shared avg (us): 10.1455
Shuffle avg (us): 8.02336
Shuffle speedup: 1.26449x
--------------------------
N=1000000 expected=5500000 shared=5500000 shuffle=5500000 PASS
Shared avg (us): 24.146
Shuffle avg (us): 17.2436
Shuffle speedup: 1.40029x
--------------------------


In [26]:
!grep -n "Shared avg" /content/warp_reduction/src/scalable_reduction.cu

304:            << "Shared avg (us): "


In [16]:
!nvidia-smi

Mon Sep 28 10:11:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [27]:
!/content/warp_reduction/src/scalable_reduction

N=32 expected=168 shared=168 shuffle=168 PASS
Shared avg (us): 8.7863
Shuffle avg (us): 7.90118
Shuffle speedup: 1.11202x
--------------------------
N=256 expected=1396 shared=1396 shuffle=1396 PASS
Shared avg (us): 8.78592
Shuffle avg (us): 7.98694
Shuffle speedup: 1.10004x
--------------------------
N=1000 expected=5500 shared=5500 shuffle=5500 PASS
Shared avg (us): 8.85773
Shuffle avg (us): 7.99725
Shuffle speedup: 1.1076x
--------------------------
N=65536 expected=360436 shared=360436 shuffle=360436 PASS
Shared avg (us): 13.3269
Shuffle avg (us): 10.2717
Shuffle speedup: 1.29743x
--------------------------
N=1000000 expected=5500000 shared=5500000 shuffle=5500000 PASS
Shared avg (us): 32.3921
Shuffle avg (us): 22.7787
Shuffle speedup: 1.42204x
--------------------------


In [28]:
!grep -n "Shared avg" /content/warp_reduction/src/scalable_reduction.cu

304:            << "Shared avg (us): "


In [29]:
%%writefile /content/warp_reduction/src/scalable_reduction.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <algorithm>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) << "\n"; \
        return 1; \
    } \
} while (0)

constexpr int THREADS = 256;
constexpr int MAX_BLOCKS = 1024;


// ==========================================
// 1. SHARED-MEMORY REDUCTION
// ==========================================

__global__ void reduceShared(
    const int* input,
    int* partial,
    int n
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;

    // Grid-stride loop
    for (; i < n; i += step) {
        sum += input[i];
    }

    s[tid] = sum;
    __syncthreads();

    // Tree reduction in shared memory
    for (int stride = blockDim.x / 2;
         stride > 0;
         stride /= 2) {

        if (tid < stride) {
            s[tid] += s[tid + stride];
        }

        __syncthreads();
    }

    if (tid == 0) {
        partial[blockIdx.x] = s[0];
    }
}


// ==========================================
// 2. WARP-SHUFFLE REDUCTION
// ==========================================

__global__ void reduceShuffle(
    const int* input,
    int* partial,
    int n
) {
    __shared__ int warpSums[THREADS / 32];

    int tid = threadIdx.x;
    int lane = tid % 32;
    int warp = tid / 32;

    int i = blockIdx.x * blockDim.x + tid;
    int step = gridDim.x * blockDim.x;

    int sum = 0;

    // Grid-stride loop
    for (; i < n; i += step) {
        sum += input[i];
    }

    unsigned int mask = 0xffffffff;

    // Reduce inside each warp
    for (int offset = 16;
         offset > 0;
         offset /= 2) {

        sum += __shfl_down_sync(
            mask, sum, offset
        );
    }

    // One lane per warp stores its result
    if (lane == 0) {
        warpSums[warp] = sum;
    }

    __syncthreads();

    // First warp reduces the warp totals
    if (warp == 0) {

        int total = (lane < THREADS / 32)
            ? warpSums[lane]
            : 0;

        for (int offset = 16;
             offset > 0;
             offset /= 2) {

            total += __shfl_down_sync(
                mask, total, offset
            );
        }

        if (lane == 0) {
            partial[blockIdx.x] = total;
        }
    }
}


// ==========================================
// 3. FINAL REDUCTION KERNEL
// ==========================================

__global__ void finishShared(
    const int* partial,
    int* output,
    int count
) {
    __shared__ int s[THREADS];

    int tid = threadIdx.x;
    int sum = 0;

    // Each thread may process multiple partials
    for (int i = tid;
         i < count;
         i += blockDim.x) {

        sum += partial[i];
    }

    s[tid] = sum;
    __syncthreads();

    // Final tree reduction
    for (int stride = blockDim.x / 2;
         stride > 0;
         stride /= 2) {

        if (tid < stride) {
            s[tid] += s[tid + stride];
        }

        __syncthreads();
    }

    if (tid == 0) {
        output[0] = s[0];
    }
}


// ==========================================
// 4. MAIN PROGRAM
// ==========================================

int main() {

    std::vector<int> sizes = {
        32,
        256,
        1000,
        65536,
        1000000
    };

    const int ITERS = 500;
    const int WARMUP = 50;

    for (int n : sizes) {

        std::cout << "\n-----------------------------\n";
        std::cout << "Input size: " << n << "\n";


        // ----------------------------------
        // A. CREATE INPUT ON CPU
        // ----------------------------------

        std::vector<int> h_input(n);

        for (int i = 0; i < n; i++) {
            h_input[i] = (i % 10) + 1;
        }

        long long expected = std::accumulate(
            h_input.begin(),
            h_input.end(),
            0LL
        );


        // ----------------------------------
        // B. CALCULATE GRID SIZE
        // ----------------------------------

        int blocks = std::min(
            (n + THREADS - 1) / THREADS,
            MAX_BLOCKS
        );


        // ----------------------------------
        // C. ALLOCATE GPU MEMORY
        // ----------------------------------

        int *d_input = nullptr;
        int *d_partial = nullptr;
        int *d_output = nullptr;

        CHECK(cudaMalloc(
            &d_input,
            n * sizeof(int)
        ));

        CHECK(cudaMalloc(
            &d_partial,
            blocks * sizeof(int)
        ));

        CHECK(cudaMalloc(
            &d_output,
            sizeof(int)
        ));


        // ----------------------------------
        // D. COPY INPUT TO GPU
        // ----------------------------------

        CHECK(cudaMemcpy(
            d_input,
            h_input.data(),
            n * sizeof(int),
            cudaMemcpyHostToDevice
        ));


        // ----------------------------------
        // E. CORRECTNESS TEST: SHARED
        // ----------------------------------

        int sharedResult = 0;
        int shuffleResult = 0;

        reduceShared<<<blocks, THREADS>>>(
            d_input,
            d_partial,
            n
        );
        CHECK(cudaGetLastError());

        finishShared<<<1, THREADS>>>(
            d_partial,
            d_output,
            blocks
        );
        CHECK(cudaGetLastError());

        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(
            &sharedResult,
            d_output,
            sizeof(int),
            cudaMemcpyDeviceToHost
        ));


        // ----------------------------------
        // F. CORRECTNESS TEST: SHUFFLE
        // ----------------------------------

        reduceShuffle<<<blocks, THREADS>>>(
            d_input,
            d_partial,
            n
        );
        CHECK(cudaGetLastError());

        finishShared<<<1, THREADS>>>(
            d_partial,
            d_output,
            blocks
        );
        CHECK(cudaGetLastError());

        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(
            &shuffleResult,
            d_output,
            sizeof(int),
            cudaMemcpyDeviceToHost
        ));


        // ----------------------------------
        // G. PRINT CORRECTNESS RESULTS
        // ----------------------------------

        bool passed =
            (sharedResult == expected) &&
            (shuffleResult == expected);

        std::cout
            << "N=" << n
            << " expected=" << expected
            << " shared=" << sharedResult
            << " shuffle=" << shuffleResult
            << " "
            << (passed ? "PASS" : "FAIL")
            << "\n";


        // ----------------------------------
        // H. WARM-UP
        // ----------------------------------

        for (int i = 0; i < WARMUP; i++) {

            reduceShared<<<blocks, THREADS>>>(
                d_input,
                d_partial,
                n
            );

            finishShared<<<1, THREADS>>>(
                d_partial,
                d_output,
                blocks
            );

            reduceShuffle<<<blocks, THREADS>>>(
                d_input,
                d_partial,
                n
            );

            finishShared<<<1, THREADS>>>(
                d_partial,
                d_output,
                blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());


        // ----------------------------------
        // I. CREATE CUDA EVENTS
        // ----------------------------------

        cudaEvent_t start, stop;

        CHECK(cudaEventCreate(&start));
        CHECK(cudaEventCreate(&stop));


        // ----------------------------------
        // J. TIME SHARED-MEMORY METHOD
        // ----------------------------------

        CHECK(cudaEventRecord(start));

        for (int i = 0; i < ITERS; i++) {

            reduceShared<<<blocks, THREADS>>>(
                d_input,
                d_partial,
                n
            );

            finishShared<<<1, THREADS>>>(
                d_partial,
                d_output,
                blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaEventRecord(stop));
        CHECK(cudaEventSynchronize(stop));

        float sharedMs = 0.0f;

        CHECK(cudaEventElapsedTime(
            &sharedMs,
            start,
            stop
        ));


        // ----------------------------------
        // K. TIME WARP-SHUFFLE METHOD
        // ----------------------------------

        CHECK(cudaEventRecord(start));

        for (int i = 0; i < ITERS; i++) {

            reduceShuffle<<<blocks, THREADS>>>(
                d_input,
                d_partial,
                n
            );

            finishShared<<<1, THREADS>>>(
                d_partial,
                d_output,
                blocks
            );
        }

        CHECK(cudaGetLastError());
        CHECK(cudaEventRecord(stop));
        CHECK(cudaEventSynchronize(stop));

        float shuffleMs = 0.0f;

        CHECK(cudaEventElapsedTime(
            &shuffleMs,
            start,
            stop
        ));


        // ----------------------------------
        // L. PRINT TIMINGS
        // ----------------------------------

        float sharedUs =
            sharedMs * 1000.0f / ITERS;

        float shuffleUs =
            shuffleMs * 1000.0f / ITERS;

        std::cout
            << "Shared avg (us): "
            << sharedUs << "\n";

        std::cout
            << "Shuffle avg (us): "
            << shuffleUs << "\n";

        if (shuffleUs > 0.0f) {
            std::cout
                << "Speedup (shared/shuffle): "
                << sharedUs / shuffleUs
                << "x\n";
        }


        // ----------------------------------
        // M. CLEANUP
        // ----------------------------------

        CHECK(cudaEventDestroy(start));
        CHECK(cudaEventDestroy(stop));

        CHECK(cudaFree(d_input));
        CHECK(cudaFree(d_partial));
        CHECK(cudaFree(d_output));
    }

    return 0;
}

Overwriting /content/warp_reduction/src/scalable_reduction.cu


In [30]:
!nvidia-smi

Mon Sep 28 11:37:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [31]:
!nvcc /content/warp_reduction/src/scalable_reduction.cu -o /content/warp_reduction/src/scalable_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [32]:
!/content/warp_reduction/src/scalable_reduction


-----------------------------
Input size: 32
N=32 expected=168 shared=168 shuffle=168 PASS
Shared avg (us): 6.87565
Shuffle avg (us): 6.16922
Speedup (shared/shuffle): 1.11451x

-----------------------------
Input size: 256
N=256 expected=1396 shared=1396 shuffle=1396 PASS
Shared avg (us): 6.81165
Shuffle avg (us): 6.1968
Speedup (shared/shuffle): 1.09922x

-----------------------------
Input size: 1000
N=1000 expected=5500 shared=5500 shuffle=5500 PASS
Shared avg (us): 9.37466
Shuffle avg (us): 6.27066
Speedup (shared/shuffle): 1.495x

-----------------------------
Input size: 65536
N=65536 expected=360436 shared=360436 shuffle=360436 PASS
Shared avg (us): 10.1359
Shuffle avg (us): 8.0071
Speedup (shared/shuffle): 1.26586x

-----------------------------
Input size: 1000000
N=1000000 expected=5500000 shared=5500000 shuffle=5500000 PASS
Shared avg (us): 24.0783
Shuffle avg (us): 17.2278
Speedup (shared/shuffle): 1.39764x


In [33]:
!mkdir -p /content/warp_reduction/results

!nvcc /content/warp_reduction/src/scalable_reduction.cu \
-o /content/warp_reduction/src/scalable_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Wrap-Level-Reduction/warp_reduction"
)

SRC = PROJECT / "src"
RESULTS = PROJECT / "results"

SRC.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Source folder exists:", SRC.exists())
print("Results folder:", RESULTS)

Project: /content/drive/MyDrive/Wrap-Level-Reduction/warp_reduction
Source folder exists: True
Results folder: /content/drive/MyDrive/Wrap-Level-Reduction/warp_reduction/results


In [36]:
import shutil

old_file = Path("/content/warp_reduction/src/scalable_reduction.cu")
new_file = SRC / "scalable_reduction.cu"

if old_file.exists() and not new_file.exists():
    shutil.copy2(old_file, new_file)
    print("Copied CUDA source to Drive.")
else:
    print("Drive source already exists or old source was not found.")

Copied CUDA source to Drive.


In [37]:
import subprocess

gpu_info = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

cuda_info = subprocess.run(
    ["nvcc", "--version"],
    capture_output=True,
    text=True
)

(RESULTS / "environment.txt").write_text(
    "GPU INFORMATION\n"
    + gpu_info.stdout
    + "\nCUDA COMPILER\n"
    + cuda_info.stdout
)

print(gpu_info.stdout)
print(cuda_info.stdout)

Mon Sep 28 11:38:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [38]:
source = Path("/content/warp_reduction/src/scalable_reduction.cu")
executable = SRC / "scalable_reduction"

assert source.exists(), f"Missing source file: {source}"

compile_result = subprocess.run(
    [
        "nvcc",
        str(source),
        "-o",
        str(executable)
    ],
    capture_output=True,
    text=True
)

print(compile_result.stdout)
print(compile_result.stderr)

if compile_result.returncode != 0:
    raise RuntimeError("Compilation failed. Fix the errors before benchmarking.")

print("Compilation successful.")


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

Compilation successful.


In [39]:
source = SRC / "scalable_reduction.cu"
executable = SRC / "scalable_reduction"

assert source.exists(), f"Missing source file: {source}"

compile_result = subprocess.run(
    [
        "nvcc",
        str(source),
        "-o",
        str(executable)
    ],
    capture_output=True,
    text=True
)

print(compile_result.stdout)
print(compile_result.stderr)

if compile_result.returncode != 0:
    raise RuntimeError("Compilation failed. Fix the errors before benchmarking.")

print("Compilation successful.")


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

Compilation successful.


In [40]:
import time
from datetime import datetime

RUNS = 5
raw_file = RESULTS / "benchmark_output.txt"

all_output = []

for run_number in range(1, RUNS + 1):
    print(f"Starting benchmark run {run_number}/{RUNS}...")

    start_time = datetime.now().astimezone().isoformat()

    result = subprocess.run(
        [str(executable)],
        capture_output=True,
        text=True
    )

    end_time = datetime.now().astimezone().isoformat()

    run_text = (
        f"\n{'=' * 60}\n"
        f"RUN {run_number}\n"
        f"Started: {start_time}\n"
        f"Finished: {end_time}\n"
        f"Exit code: {result.returncode}\n"
        f"{'=' * 60}\n"
        + result.stdout
        + "\nSTDERR:\n"
        + result.stderr
    )

    all_output.append(run_text)

    if result.returncode != 0:
        print(run_text)
        raise RuntimeError(f"Benchmark run {run_number} failed.")

    print(f"Run {run_number} completed.")

    time.sleep(2)

raw_file.write_text("".join(all_output))

print("\nAll runs saved to:")
print(raw_file)

Starting benchmark run 1/5...
Run 1 completed.
Starting benchmark run 2/5...
Run 2 completed.
Starting benchmark run 3/5...
Run 3 completed.
Starting benchmark run 4/5...
Run 4 completed.
Starting benchmark run 5/5...
Run 5 completed.

All runs saved to:
/content/drive/MyDrive/Wrap-Level-Reduction/warp_reduction/results/benchmark_output.txt


In [41]:
import re
import pandas as pd

text_output = raw_file.read_text()

# Each size's correctness line
correctness_pattern = re.compile(
    r"N=(\d+)\s+expected=(-?\d+)\s+"
    r"shared=(-?\d+)\s+shuffle=(-?\d+)\s+"
    r"(PASS|FAIL)"
)

# Timing lines immediately follow each size's correctness line
timing_pattern = re.compile(
    r"Shared avg \(us\):\s*([0-9.eE+-]+)\s*"
    r"Shuffle avg \(us\):\s*([0-9.eE+-]+)"
)

# Split output into individual run sections
run_sections = re.split(
    r"={10,}\s*RUN\s+(\d+)\s*",
    text_output
)

rows = []

for i in range(1, len(run_sections), 2):
    run_number = int(run_sections[i])
    section = run_sections[i + 1]

    checks = list(correctness_pattern.finditer(section))
    timings = list(timing_pattern.finditer(section))

    if len(checks) != 5 or len(timings) != 5:
        raise ValueError(
            f"Run {run_number}: expected 5 correctness and "
            f"5 timing records; got {len(checks)} and {len(timings)}."
        )

    for check, timing in zip(checks, timings):
        n, expected, shared, shuffle, status = check.groups()
        shared_us, shuffle_us = timing.groups()

        shared_us = float(shared_us)
        shuffle_us = float(shuffle_us)

        rows.append({
            "run": run_number,
            "input_size": int(n),
            "expected": int(expected),
            "shared_result": int(shared),
            "shuffle_result": int(shuffle),
            "correctness": status,
            "shared_us": shared_us,
            "shuffle_us": shuffle_us,
            "speedup_shared_over_shuffle": (
                shared_us / shuffle_us
                if shuffle_us > 0 else None
            )
        })

raw_df = pd.DataFrame(rows).sort_values(
    ["input_size", "run"]
)

raw_csv = RESULTS / "benchmark_raw.csv"
raw_df.to_csv(raw_csv, index=False)

print(raw_df.to_string(index=False))
print("\nSaved:", raw_csv)

 run  input_size  expected  shared_result  shuffle_result correctness  shared_us  shuffle_us  speedup_shared_over_shuffle
   1          32       168            168             168        PASS   13.49220     8.21978                     1.641431
   2          32       168            168             168        PASS   10.40800     9.23565                     1.126937
   3          32       168            168             168        PASS    6.96845     6.30573                     1.105098
   4          32       168            168             168        PASS    8.92102     8.01587                     1.112920
   5          32       168            168             168        PASS    6.96915     6.28301                     1.109206
   1         256      1396           1396            1396        PASS    8.95386     8.04038                     1.113612
   2         256      1396           1396            1396        PASS   10.40360     9.31571                     1.116780
   3         256      13

In [42]:
import re
import pandas as pd

text_output = raw_file.read_text()

run_sections = re.split(
    r"={10,}\s*RUN\s+(\d+)\s*",
    text_output
)

rows = []

for i in range(1, len(run_sections), 2):
    run_number = int(run_sections[i])
    section = run_sections[i + 1]

    checks = list(re.finditer(
        r"N=(\d+)\s+expected=(-?\d+)\s+"
        r"shared=(-?\d+)\s+shuffle=(-?\d+)\s+"
        r"(PASS|FAIL)",
        section
    ))

    shared_times = re.findall(
        r"Shared avg \(us\):\s*([0-9.eE+-]+)",
        section
    )

    shuffle_times = re.findall(
        r"Shuffle avg \(us\):\s*([0-9.eE+-]+)",
        section
    )

    if not (
        len(checks) == 5
        and len(shared_times) == 5
        and len(shuffle_times) == 5
    ):
        raise ValueError(
            f"Run {run_number}: expected 5 of each record. "
            f"Checks={len(checks)}, shared={len(shared_times)}, "
            f"shuffle={len(shuffle_times)}"
        )

    for j, check in enumerate(checks):
        n, expected, shared, shuffle, status = check.groups()

        shared_us = float(shared_times[j])
        shuffle_us = float(shuffle_times[j])

        rows.append({
            "run": run_number,
            "input_size": int(n),
            "expected": int(expected),
            "shared_result": int(shared),
            "shuffle_result": int(shuffle),
            "correctness": status,
            "shared_us": shared_us,
            "shuffle_us": shuffle_us,
            "speedup_shared_over_shuffle": (
                shared_us / shuffle_us
                if shuffle_us > 0 else None
            )
        })

raw_df = pd.DataFrame(rows).sort_values(
    ["input_size", "run"]
)

raw_csv = RESULTS / "benchmark_raw.csv"
raw_df.to_csv(raw_csv, index=False)

print(raw_df.to_string(index=False))
print("\nSaved:", raw_csv)

 run  input_size  expected  shared_result  shuffle_result correctness  shared_us  shuffle_us  speedup_shared_over_shuffle
   1          32       168            168             168        PASS   13.49220     8.21978                     1.641431
   2          32       168            168             168        PASS   10.40800     9.23565                     1.126937
   3          32       168            168             168        PASS    6.96845     6.30573                     1.105098
   4          32       168            168             168        PASS    8.92102     8.01587                     1.112920
   5          32       168            168             168        PASS    6.96915     6.28301                     1.109206
   1         256      1396           1396            1396        PASS    8.95386     8.04038                     1.113612
   2         256      1396           1396            1396        PASS   10.40360     9.31571                     1.116780
   3         256      13

In [43]:
summary = (
    raw_df.groupby("input_size")
    .agg(
        runs=("run", "count"),
        all_pass=("correctness", lambda x: (x == "PASS").all()),
        median_shared_us=("shared_us", "median"),
        median_shuffle_us=("shuffle_us", "median"),
        median_speedup=("speedup_shared_over_shuffle", "median"),
        min_speedup=("speedup_shared_over_shuffle", "min"),
        max_speedup=("speedup_shared_over_shuffle", "max")
    )
    .reset_index()
)

summary["speedup_from_median_times"] = (
    summary["median_shared_us"]
    / summary["median_shuffle_us"]
)

summary["latency_reduction_percent"] = (
    (
        summary["median_shared_us"]
        - summary["median_shuffle_us"]
    )
    / summary["median_shared_us"]
    * 100
)

summary_csv = RESULTS / "benchmark_summary.csv"
summary.to_csv(summary_csv, index=False)

print(summary.round(4).to_string(index=False))
print("\nSaved:", summary_csv)

 input_size  runs  all_pass  median_shared_us  median_shuffle_us  median_speedup  min_speedup  max_speedup  speedup_from_median_times  latency_reduction_percent
         32     5      True            8.9210             8.0159          1.1129       1.1051       1.6414                     1.1129                    10.1463
        256     5      True            8.9211             8.0404          1.1050       1.0991       1.1168                     1.1095                     9.8722
       1000     5      True            8.9661             8.1169          1.1046       1.0961       1.1195                     1.1046                     9.4721
      65536     5      True           13.6274             9.6191          1.2931       1.2631       1.4356                     1.4167                    29.4132
    1000000     5      True           29.5940            20.9838          1.4103       1.3972       1.4251                     1.4103                    29.0944

Saved: /content/drive/MyDrive/Wra

In [44]:
summary = (
    raw_df.groupby("input_size")
    .agg(
        runs=("run", "count"),
        all_pass=("correctness", lambda x: (x == "PASS").all()),
        median_shared_us=("shared_us", "median"),
        median_shuffle_us=("shuffle_us", "median"),
        median_speedup=("speedup_shared_over_shuffle", "median"),
        min_speedup=("speedup_shared_over_shuffle", "min"),
        max_speedup=("speedup_shared_over_shuffle", "max")
    )
    .reset_index()
)

summary["speedup_from_median_times"] = (
    summary["median_shared_us"]
    / summary["median_shuffle_us"]
)

summary["latency_reduction_percent"] = (
    (
        summary["median_shared_us"]
        - summary["median_shuffle_us"]
    )
    / summary["median_shared_us"]
    * 100
)

summary_csv = RESULTS / "benchmark_summary.csv"
summary.to_csv(summary_csv, index=False)

print(summary.round(4).to_string(index=False))
print("\nSaved:", summary_csv)

 input_size  runs  all_pass  median_shared_us  median_shuffle_us  median_speedup  min_speedup  max_speedup  speedup_from_median_times  latency_reduction_percent
         32     5      True            8.9210             8.0159          1.1129       1.1051       1.6414                     1.1129                    10.1463
        256     5      True            8.9211             8.0404          1.1050       1.0991       1.1168                     1.1095                     9.8722
       1000     5      True            8.9661             8.1169          1.1046       1.0961       1.1195                     1.1046                     9.4721
      65536     5      True           13.6274             9.6191          1.2931       1.2631       1.4356                     1.4167                    29.4132
    1000000     5      True           29.5940            20.9838          1.4103       1.3972       1.4251                     1.4103                    29.0944

Saved: /content/drive/MyDrive/Wra

In [45]:
print("RESULTS FOLDER CONTENTS")
for file in sorted(RESULTS.iterdir()):
    if file.is_file():
        print(f"{file.name:28} {file.stat().st_size:,} bytes")

assert len(raw_df) == 25, "Expected 25 rows (5 runs × 5 sizes)"
assert summary.shape[0] == 5, "Expected 5 summary rows"
assert raw_df["correctness"].eq("PASS").all(), (
    "At least one correctness test failed."
)

print("\nVerification successful: 25 measurements, 5 sizes, all PASS.")

RESULTS FOLDER CONTENTS
benchmark_output.txt         5,869 bytes
benchmark_raw.csv            1,717 bytes
benchmark_summary.csv        759 bytes
environment.txt              1,928 bytes

Verification successful: 25 measurements, 5 sizes, all PASS.
